In [29]:
import os

In [30]:
os.chdir("../")

In [31]:
%pwd

'd:\\Data_Science\\Projects\\Chicken-Disease-Classification-'

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import requests
from cnnClassifier import logger
import zipfile

In [3]:
@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [4]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories

In [24]:
class ConfigurationManager:
    def __init__(self, config_file_path: Path = CONFIG_FILE_PATH, params_file_path: Path = PARAMS_FILE_PATH):
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        create_directories([self.config.artifacts_root])
    
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )
        return data_ingestion_config

In [44]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            os.makedirs(os.path.dirname(self.config.local_data_file), exist_ok=True)
            response = requests.get(self.config.source_URL, stream=True)
            response.raise_for_status()  # Check if the request was successful
            with open(self.config.local_data_file, 'wb') as wf:
                for chunk in response.iter_content(chunk_size=1024):
                    wf.write(chunk)
            logger.info(f"File downloaded successfully and saved to {self.config.local_data_file}")
        else:
            logger.info(f"File already exists at {self.config.local_data_file}")


    def unzip_and_clean(self):
        """Unzips the downloaded file and removes the zip file after extraction."""

        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)
        logger.info(f"File unzipped successfully to {unzip_path}")
        

    def initiate_data_ingestion(self):
        self.download_file()
        self.unzip_and_clean()

In [48]:
try:
    config_manager = ConfigurationManager()
    data_ingestion_config = config_manager.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.initiate_data_ingestion()
except Exception as e:
    logger.exception(e)

[2026-02-15 23:57:50,034 - INFO - common - yaml file: config\config.yaml loaded successfully]


[2026-02-15 23:57:50,036 - INFO - common - yaml file: params.yaml loaded successfully]
[2026-02-15 23:57:50,037 - INFO - common - created directory at: artifacts]
[2026-02-15 23:57:54,367 - INFO - 3420179910 - File downloaded successfully and saved to artifacts/data_ingestion/data.zip]
[2026-02-15 23:57:54,581 - INFO - 3420179910 - File unzipped successfully to artifacts/data_ingestion]


In [26]:
%pwd

'd:\\Data_Science\\Projects\\Chicken-Disease-Classification-\\research'